# **Agentic Healthcare Assistant for Medical Task Automation**
### Applied Generative AI Specialisation — Capstone Project

---

**Objective:**
Build an **Agentic AI Healthcare Assistant** that autonomously handles:
- 📅 **Appointment booking** — checks doctor availability and schedules slots
- 📋 **Medical record retrieval** — summarizes patient history from PDF/Excel records
- ✏️ **Record management** — adds or updates patient notes
- 🔍 **Medical information search** — fetches disease and treatment information

**Architecture:**
- **LangGraph ReAct Agent** — plans and decomposes multi-step healthcare queries
- **5 specialized tools** — appointment, history, record update, search, summarize
- **FAISS Knowledge Base** — patient PDFs embedded with OpenAI embeddings
- **Conversational Memory** — `MemorySaver` retains context across turns
- **Evaluation** — `QAEvalChain` for response quality assessment
- **Streamlit UI** — patient/doctor dashboard with agent chat

**Dataset:** `records.xlsx` (5 patients) + 4 patient PDF reports


---
## Part 1: Agentic Healthcare Assistant System Design
---

### **Setup: Environment & Imports**
> Ensure `OPENAI_API_KEY` is set in `.env` before running.


In [ ]:
import os, sys, json, warnings
warnings.filterwarnings("ignore")
from dotenv import load_dotenv
load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found in .env"
print("Environment ready.")


In [ ]:
# ── Core ────────────────────────────────────────────────────────────────────
import pandas as pd
from datetime import datetime, timedelta
from typing import Annotated

# ── LangChain / LangGraph ────────────────────────────────────────────────────
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.tools import tool
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain.evaluation import QAEvalChain

from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

# ── Misc ─────────────────────────────────────────────────────────────────────
import wikipedia
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

print("All libraries imported.")


---
### **Step 1: Agent Planning and Goal Decomposition**

The agent uses a **ReAct (Reason + Act)** loop to:
1. **Interpret** the user query (e.g., *"Book a nephrologist for my father"*)
2. **Decompose** it into sequential sub-goals (identify patient → check schedule → book → search)
3. **Select** the appropriate tool for each sub-goal
4. **Execute** and synthesize results into a coherent response

The LLM acts as the planner; tools act as executors.


In [ ]:
# Initialize LLM — the agent's reasoning engine
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("LLM initialized:", llm.model_name)


---
### **Step 2: Tool and Memory Setup**

**Data sources loaded:**
- `records.xlsx` → in-memory patient database
- `sample_report_*.pdf` + `sample_patient.pdf` → embedded in FAISS for semantic retrieval

**Mock systems simulated:**
- Doctor schedule (specialty → date → available slots)
- Appointment booking registry


In [ ]:
# ── Patient Database (Excel) ─────────────────────────────────────────────────
DATA_DIR = "../Datasets_New/Agentic Healthcare Assistant for Medical Task Automation/"

patient_df = pd.read_excel(DATA_DIR + "records.xlsx")
patient_db = {}
for _, row in patient_df.iterrows():
    name = str(row.get("Name", "")).strip()
    if name:
        patient_db[name.lower()] = {
            "name"    : name,
            "age"     : row.get("Age"),
            "gender"  : row.get("Gender"),
            "phone"   : str(row.get("Phone_number", "")),
            "email"   : str(row.get("Email", "")),
            "address" : str(row.get("Address", "")),
            "summary" : str(row.get("Summary", "No summary available.")),
        }

print(f"Loaded {len(patient_db)} patients: {list(patient_db.keys())}")


In [ ]:
# ── FAISS Knowledge Base from Patient PDFs ───────────────────────────────────
pdf_files = [
    "sample_patient.pdf",
    "sample_report_anjali.pdf",
    "sample_report_david.pdf",
    "sample_report_ramesh.pdf",
]

all_docs = []
for fname in pdf_files:
    path = DATA_DIR + fname
    try:
        loader = PyPDFLoader(path)
        pages  = loader.load()
        for page in pages:
            page.metadata["source"] = fname
        all_docs.extend(pages)
        print(f"  Loaded {len(pages)} page(s) from {fname}")
    except Exception as e:
        print(f"  Could not load {fname}: {e}")

splitter  = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=64)
split_docs = splitter.split_documents(all_docs)
print(f"\nTotal chunks: {len(split_docs)}")


In [ ]:
# ── Create FAISS vectorstore ──────────────────────────────────────────────────
embeddings  = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(split_docs, embeddings)
retriever   = vectorstore.as_retriever(search_kwargs={"k": 4})
print(f"FAISS index built: {vectorstore.index.ntotal} vectors")


In [ ]:
# ── Mock Doctor Schedule ──────────────────────────────────────────────────────
# {specialty: {date_str: [available_slots]}}
# Slots get removed once booked

today = datetime.today()
DOCTOR_SCHEDULE = {
    "general physician": {
        (today + timedelta(days=1)).strftime("%Y-%m-%d"): ["09:00","10:00","11:00","14:00"],
        (today + timedelta(days=2)).strftime("%Y-%m-%d"): ["09:00","10:30","15:00"],
    },
    "cardiologist": {
        (today + timedelta(days=1)).strftime("%Y-%m-%d"): ["10:00","13:00"],
        (today + timedelta(days=3)).strftime("%Y-%m-%d"): ["09:00","11:00","14:00"],
    },
    "nephrologist": {
        (today + timedelta(days=2)).strftime("%Y-%m-%d"): ["09:30","11:30","14:00"],
        (today + timedelta(days=4)).strftime("%Y-%m-%d"): ["10:00","13:00"],
    },
    "endocrinologist": {
        (today + timedelta(days=1)).strftime("%Y-%m-%d"): ["09:00","11:00"],
        (today + timedelta(days=2)).strftime("%Y-%m-%d"): ["10:00","14:00","15:30"],
    },
    "pulmonologist": {
        (today + timedelta(days=1)).strftime("%Y-%m-%d"): ["09:00","10:00","14:00"],
        (today + timedelta(days=3)).strftime("%Y-%m-%d"): ["11:00","15:00"],
    },
}

BOOKED_APPOINTMENTS = []  # log of confirmed bookings

print("Doctor schedule initialized.")
for spec, dates in DOCTOR_SCHEDULE.items():
    total_slots = sum(len(s) for s in dates.values())
    print(f"  {spec.title()}: {total_slots} available slots")


---
### **Step 3: Prompt Engineering and Task Chaining — Tool Definitions**

Each tool is a structured LangChain `@tool` with a clear docstring that guides
the agent's planner when deciding which tool to call.


In [ ]:
@tool
def get_patient_history(patient_name: str) -> str:
    """
    Retrieve structured patient information and medical summary from the database.
    Use this when asked about a patient's background, age, diagnosis, or history.
    Input: patient's full name (case-insensitive).
    """
    key = patient_name.strip().lower()
    # Fuzzy match
    match = next((v for k, v in patient_db.items() if key in k or k in v["name"].lower()), None)
    if not match:
        return f"No patient record found for '{patient_name}'. Available patients: {[v['name'] for v in patient_db.values()]}"
    return (
        f"Patient Record — {match['name']}\n"
        f"Age: {match['age']} | Gender: {match['gender']}\n"
        f"Phone: {match['phone']} | Address: {match['address']}\n"
        f"Medical Summary: {match['summary']}"
    )

print("Tool defined: get_patient_history")


In [ ]:
@tool
def retrieve_medical_documents(patient_name: str) -> str:
    """
    Retrieve detailed clinical notes, lab results, diagnoses, and treatment plans
    from patient PDF documents using semantic search.
    Use this for in-depth medical history or specific clinical details.
    Input: patient's name or relevant medical query.
    """
    docs = retriever.invoke(patient_name)
    if not docs:
        return f"No clinical documents found for '{patient_name}'."
    results = []
    for doc in docs:
        results.append(f"[{doc.metadata.get('source','doc')}]\n{doc.page_content.strip()}")
    return "\n\n---\n\n".join(results)

print("Tool defined: retrieve_medical_documents")


In [ ]:
@tool
def book_appointment(patient_name: str, specialty: str, preferred_date: str = "") -> str:
    """
    Book a medical appointment for a patient with a specialist.
    Checks real-time availability and confirms the earliest available slot.
    Input: patient_name, specialty (e.g. 'cardiologist', 'nephrologist'),
           optional preferred_date (YYYY-MM-DD format).
    """
    spec_key = specialty.strip().lower()

    # Find closest specialty match
    matched_spec = next(
        (s for s in DOCTOR_SCHEDULE if spec_key in s or s in spec_key), None
    )
    if not matched_spec:
        return (f"No {specialty} available in our system. "
                f"Available specialties: {list(DOCTOR_SCHEDULE.keys())}")

    slots = DOCTOR_SCHEDULE[matched_spec]

    # If preferred date given, try that first
    if preferred_date and preferred_date in slots and slots[preferred_date]:
        date, slot = preferred_date, slots[preferred_date][0]
    else:
        # Find earliest available
        future = {d: s for d, s in sorted(slots.items()) if s}
        if not future:
            return f"No available slots for {matched_spec}. Please try another specialty."
        date, slot = next(iter(future.items()))
        slot = slot[0]

    # Book the slot (remove from schedule)
    DOCTOR_SCHEDULE[matched_spec][date].remove(slot)
    BOOKED_APPOINTMENTS.append({
        "patient": patient_name,
        "specialty": matched_spec,
        "date": date,
        "time": slot,
        "booked_at": datetime.now().strftime("%Y-%m-%d %H:%M")
    })

    return (f"✅ Appointment Confirmed!\n"
            f"Patient   : {patient_name}\n"
            f"Specialist: {matched_spec.title()}\n"
            f"Date      : {date}\n"
            f"Time      : {slot}\n"
            f"Reference : APT-{len(BOOKED_APPOINTMENTS):04d}")

print("Tool defined: book_appointment")


In [ ]:
@tool
def update_patient_record(patient_name: str, update_notes: str) -> str:
    """
    Add or update a patient's medical notes or summary in the database.
    Use this when the user wants to record a new diagnosis, treatment, or observation.
    Input: patient_name, update_notes (the new information to add).
    """
    key = patient_name.strip().lower()
    match_key = next((k for k in patient_db if key in k or k in patient_name.lower()), None)
    if not match_key:
        return f"Patient '{patient_name}' not found. Cannot update record."

    old_summary = patient_db[match_key]["summary"]
    timestamp   = datetime.now().strftime("%Y-%m-%d %H:%M")
    new_summary = f"{old_summary}\n[Updated {timestamp}]: {update_notes}"
    patient_db[match_key]["summary"] = new_summary

    return (f"✅ Record updated for {patient_db[match_key]['name']}\n"
            f"Added note: {update_notes}")

print("Tool defined: update_patient_record")


In [ ]:
@tool
def search_medical_info(query: str) -> str:
    """
    Search for up-to-date medical information about diseases, symptoms,
    treatments, or medications from trusted medical knowledge sources.
    Use this when asked about conditions, treatment options, or drug interactions.
    Input: a medical search query (e.g., 'chronic kidney disease treatment').
    """
    try:
        # Try direct Wikipedia medical search
        result = wikipedia.summary(query, sentences=6, auto_suggest=True)
        return f"Medical Information — '{query}':\n\n{result}"
    except wikipedia.exceptions.DisambiguationError as e:
        # Try the first suggested option
        try:
            result = wikipedia.summary(e.options[0], sentences=6)
            return f"Medical Information — '{e.options[0]}':\n\n{result}"
        except:
            return f"Multiple topics found for '{query}'. Try a more specific term."
    except wikipedia.exceptions.PageError:
        return f"No Wikipedia article found for '{query}'. Please try a different search term."
    except Exception as e:
        return f"Search error for '{query}': {str(e)}"

print("Tool defined: search_medical_info")


In [ ]:
@tool
def list_booked_appointments(filter_patient: str = "") -> str:
    """
    List all booked appointments. Optionally filter by patient name.
    Use this to check existing bookings or appointment status.
    Input: optional patient name filter (leave empty for all appointments).
    """
    if not BOOKED_APPOINTMENTS:
        return "No appointments have been booked yet."

    filtered = [
        a for a in BOOKED_APPOINTMENTS
        if not filter_patient or filter_patient.lower() in a["patient"].lower()
    ]
    if not filtered:
        return f"No appointments found for '{filter_patient}'."

    lines = ["Booked Appointments:"]
    for a in filtered:
        lines.append(
            f"  [{a['date']} {a['time']}] {a['patient']} → {a['specialty'].title()} "
            f"(Ref: APT-{BOOKED_APPOINTMENTS.index(a)+1:04d})"
        )
    return "\n".join(lines)

print("Tool defined: list_booked_appointments")


---
### **Step 4: Agent Execution Flow**

Build the **LangGraph ReAct agent** with:
- All 5 healthcare tools
- A detailed system prompt guiding healthcare behaviour
- `MemorySaver` for multi-turn conversational memory


In [ ]:
SYSTEM_PROMPT = """You are HealthAgent, an intelligent Agentic Healthcare Assistant
for a multi-specialty clinic. You help patients, their families, and medical staff by:

1. Retrieving and summarizing patient medical histories
2. Booking appointments with the right specialist
3. Updating patient records with new clinical notes
4. Searching for accurate medical information on diseases and treatments

Guidelines:
- Always verify the patient exists before booking or updating records
- For appointment requests, identify the correct medical specialty from symptoms/conditions
- When searching medical info, provide clear, actionable summaries
- Be compassionate and professional in all responses
- For multi-step requests, complete ALL steps before summarizing results
- Never fabricate patient data — use the tools to retrieve real information
"""

tools = [
    get_patient_history,
    retrieve_medical_documents,
    book_appointment,
    update_patient_record,
    search_medical_info,
    list_booked_appointments,
]

memory = MemorySaver()

agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=SYSTEM_PROMPT,
    checkpointer=memory
)

print("HealthAgent created with LangGraph ReAct.")
print(f"Tools registered: {[t.name for t in tools]}")


In [ ]:
def run_agent(query: str, session_id: str = "default", verbose: bool = True) -> str:
    """Run the HealthAgent and display the full reasoning trace."""
    config = {"configurable": {"thread_id": session_id}}
    result = agent.invoke({"messages": [HumanMessage(content=query)]}, config=config)

    if verbose:
        print("=" * 65)
        print(f"USER: {query}")
        print("-" * 65)
        for msg in result["messages"]:
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"[AGENT → TOOL] {tc['name']}({json.dumps(tc['args'])[:120]})")
            elif msg.type == "tool":
                content_preview = str(msg.content)[:200].replace("\n", " ")
                print(f"[TOOL RESULT]  {content_preview}...")
            elif msg.type == "ai" and msg.content:
                print(f"\nHEALTHAGENT: {msg.content}")
        print("=" * 65)

    final = next(
        (m.content for m in reversed(result["messages"]) if m.type == "ai" and m.content),
        "No response generated."
    )
    return final


---
### **Sample Scenario 1: Retrieve Patient History**


In [ ]:
response = run_agent(
    "Can you retrieve the medical history for Ramesh Kulkarni?",
    session_id="scenario_1"
)


### **Sample Scenario 2: Book an Appointment**

In [ ]:
response = run_agent(
    "I need to book an appointment with a cardiologist for Anjali Mehra.",
    session_id="scenario_2"
)


### **Sample Scenario 3: Medical Information Search**

In [ ]:
response = run_agent(
    "What are the latest treatment options for Type 2 Diabetes?",
    session_id="scenario_3"
)


### **Sample Scenario 4: Multi-Step Complex Query**
*(The example from the problem statement — kidney disease + nephrologist booking)*


In [ ]:
response = run_agent(
    """My 70-year-old father has chronic kidney disease.
I want to book a nephrologist for him under the name Ramesh Kulkarni.
Also, can you summarize the latest treatment methods for chronic kidney disease?""",
    session_id="scenario_4"
)


### **Sample Scenario 5: Update Patient Record (Multi-Turn Memory)**

In [ ]:
# Turn 1 — retrieve
run_agent("Show me David Thompson's current medical record.", session_id="scenario_5")

# Turn 2 — update (agent remembers previous context)
run_agent(
    "Update his record: new HbA1c result is 8.2%, dosage increased to metformin 1000mg BID.",
    session_id="scenario_5"
)


---
## Part 2: LLMOps — Evaluation, Monitoring & Visualization
---

---
### **Step 5: Model Evaluation with QAEvalChain**

Evaluate HealthAgent response quality using **QAEvalChain** —
the LLM grades each predicted answer as CORRECT or INCORRECT
compared to a ground-truth expected answer.


In [ ]:
eval_examples = [
    {
        "query"  : "What is Ramesh Kulkarni's medical condition?",
        "answer" : "Ramesh Kulkarni has essential hypertension (I10) and is on Telmisartan 40mg."
    },
    {
        "query"  : "What diagnosis does Anjali Mehra have?",
        "answer" : "Anjali Mehra was diagnosed with Upper Respiratory Infection (J06.9)."
    },
    {
        "query"  : "What is David Thompson's diabetes diagnosis code?",
        "answer" : "David Thompson has Type 2 Diabetes Mellitus with ICD code E11.9."
    },
    {
        "query"  : "What specialty should I book for a kidney disease patient?",
        "answer" : "A nephrologist should be booked for a kidney disease patient."
    },
    {
        "query"  : "What medication is Ramesh Kulkarni currently taking?",
        "answer" : "Ramesh Kulkarni is taking Telmisartan 40mg OD for hypertension."
    },
]

print(f"Prepared {len(eval_examples)} evaluation examples.")


In [ ]:
# Generate predictions
predictions = []
for i, ex in enumerate(eval_examples):
    pred = run_agent(ex["query"], session_id=f"eval_{i}", verbose=False)
    predictions.append({"result": pred})
    print(f"  [{i+1}] Q: {ex['query'][:70]}...")

print("\nAll predictions generated.")


In [ ]:
# Run QAEvalChain
eval_chain = QAEvalChain.from_llm(llm)
graded = eval_chain.evaluate(
    eval_examples, predictions,
    question_key="query", prediction_key="result"
)

# Results table
results_df = pd.DataFrame({
    "Question"  : [e["query"]                for e in eval_examples],
    "Expected"  : [e["answer"][:80]          for e in eval_examples],
    "Predicted" : [p["result"][:100] + "..." for p in predictions],
    "Grade"     : [g.get("results", "N/A")   for g in graded],
})
pd.set_option("display.max_colwidth", 90)
print(results_df.to_string(index=False))


In [ ]:
# Summary
grades    = [g.get("results", "") for g in graded]
correct   = sum(1 for g in grades if "CORRECT" in g.upper())
accuracy  = correct / len(grades) * 100

print(f"\n=== Evaluation Summary ===")
print(f"  Total   : {len(grades)}")
print(f"  Correct : {correct}")
print(f"  Accuracy: {accuracy:.1f}%")


---
### **Step 6: Data Visualization & Agent Activity Dashboard**


In [ ]:
sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.dpi": 120, "axes.titlesize": 12})


In [ ]:
# ── Chart 1: Patient Demographics ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Age distribution
ages   = [v["age"] for v in patient_db.values() if isinstance(v["age"], (int, float))]
names  = [v["name"].split()[0] for v in patient_db.values() if isinstance(v["age"], (int, float))]
colors = ["#2E86AB","#A23B72","#F18F01","#C73E1D","#264653"]
bars   = axes[0].bar(names, ages, color=colors)
axes[0].bar_label(bars, labels=[f"{a} yrs" for a in ages], padding=3, fontsize=9)
axes[0].set_title("Patient Age Profile", fontweight="bold")
axes[0].set_ylabel("Age")
axes[0].set_ylim(0, max(ages) + 15)

# Gender distribution
genders = [v["gender"] for v in patient_db.values() if v["gender"]]
g_counts = pd.Series(genders).value_counts()
axes[1].pie(g_counts.values, labels=g_counts.index, autopct="%1.0f%%",
            colors=["#2E86AB","#E76F51"], startangle=90)
axes[1].set_title("Patient Gender Distribution", fontweight="bold")

plt.suptitle("Patient Demographics", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("viz_01_patient_demographics.png", bbox_inches="tight")
plt.show()
print("Saved viz_01_patient_demographics.png")


In [ ]:
# ── Chart 2: Appointment Bookings ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

if BOOKED_APPOINTMENTS:
    apt_df = pd.DataFrame(BOOKED_APPOINTMENTS)

    # Bookings by specialty
    spec_counts = apt_df["specialty"].value_counts()
    axes[0].barh(spec_counts.index, spec_counts.values, color="#2A9D8F")
    axes[0].set_title("Bookings by Specialty", fontweight="bold")
    axes[0].set_xlabel("Count")

    # Bookings by patient
    pt_counts = apt_df["patient"].apply(lambda x: x.split()[0]).value_counts()
    axes[1].bar(pt_counts.index, pt_counts.values, color="#E9C46A")
    axes[1].set_title("Bookings by Patient", fontweight="bold")
    axes[1].set_ylabel("Count")
else:
    axes[0].text(0.5, 0.5, "No bookings yet", ha="center", va="center", transform=axes[0].transAxes)
    axes[1].text(0.5, 0.5, "No bookings yet", ha="center", va="center", transform=axes[1].transAxes)

plt.suptitle("Appointment Activity", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("viz_02_appointment_activity.png", bbox_inches="tight")
plt.show()
print("Saved viz_02_appointment_activity.png")


In [ ]:
# ── Chart 3: Doctor Availability ─────────────────────────────────────────────
spec_slots = {
    spec.title(): sum(len(s) for s in dates.values())
    for spec, dates in DOCTOR_SCHEDULE.items()
}
fig, ax = plt.subplots(figsize=(9, 4))
colors_avail = ["#264653","#2A9D8F","#E9C46A","#F18F01","#E76F51"]
bars = ax.bar(spec_slots.keys(), spec_slots.values(), color=colors_avail)
ax.bar_label(bars, labels=[f"{v} slots" for v in spec_slots.values()], padding=3, fontsize=9)
ax.set_title("Remaining Available Doctor Slots", fontweight="bold")
ax.set_ylabel("Available Slots")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("viz_03_doctor_availability.png", bbox_inches="tight")
plt.show()
print("Saved viz_03_doctor_availability.png")


In [ ]:
# ── Chart 4: Evaluation Results ───────────────────────────────────────────────
grade_counts = pd.Series([g.get("results","N/A") for g in graded]).value_counts()
fig, ax = plt.subplots(figsize=(5, 4))
colors_eval = ["#2A9D8F" if "CORRECT" in str(k).upper() else "#E76F51" for k in grade_counts.index]
bars = ax.bar(grade_counts.index, grade_counts.values, color=colors_eval, width=0.4)
ax.bar_label(bars, padding=3, fontsize=10)
ax.set_title("QAEvalChain — Agent Evaluation Results", fontweight="bold")
ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig("viz_04_evaluation_results.png", bbox_inches="tight")
plt.show()
print("Saved viz_04_evaluation_results.png")


---
### **Step 7: Memory and Agent Trace Inspection**

Display the agent's memory trace — all messages stored in the checkpoint
for a given session, showing the full reasoning chain.


In [ ]:
def display_memory_trace(session_id: str):
    """Inspect stored memory for a given agent session."""
    config = {"configurable": {"thread_id": session_id}}
    state  = agent.get_state(config)
    msgs   = state.values.get("messages", [])

    print(f"\n=== Memory Trace — Session: '{session_id}' ({len(msgs)} messages) ===")
    for i, msg in enumerate(msgs):
        role = msg.type.upper()
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  [{i}] {role} → TOOL CALL: {tc['name']}({str(tc['args'])[:80]})")
        elif msg.type == "tool":
            print(f"  [{i}] TOOL RESULT: {str(msg.content)[:100]}...")
        else:
            content = str(msg.content)[:120].replace("\n", " ")
            if content.strip():
                print(f"  [{i}] {role}: {content}")

display_memory_trace("scenario_4")


---
## Conclusion

**HealthAgent** delivers a fully agentic healthcare automation system:

| Component | Implementation |
|---|---|
| Agent Framework | LangGraph `create_react_agent` — ReAct planning loop |
| Patient DB | pandas (Excel) + FAISS (PDF documents) |
| Tools | 6 tools: history, documents, booking, update, search, list |
| Medical Search | Wikipedia medical knowledge retrieval |
| Memory | `MemorySaver` — multi-turn session context |
| Evaluation | `QAEvalChain` — LLM-as-judge scoring |
| Visualizations | Demographics, bookings, availability, evaluation |
| UI | Streamlit dashboard (`streamlit_app.py`) |

**Key capabilities demonstrated:**
- Single-step queries (retrieve history, search disease info)
- Multi-step autonomous workflows (identify patient → retrieve history → book → search)
- Persistent memory across conversation turns
- Live slot management with booking confirmation

---
